# LeetCode 1057 — Campus Bikes bằng Python

Link đề: https://leetcode.com/problems/campus-bikes/ · Độ khó: **Medium** · Premium (cần tài khoản LeetCode Premium để xem đề) · Chủ đề: Array (mảng), Sorting (sắp xếp), Heap (hàng đợi ưu tiên), Bucket Sort

> **Lưu ý:** đề bài của bài Premium này được viết theo hiểu biết và theo bản mô tả đã có; nên đối chiếu lại nếu bạn có tài khoản Premium.

Skeleton LeetCode:

```python
class Solution:
    def assignBikes(self, workers: List[List[int]], bikes: List[List[int]]) -> List[int]:
        ...
```

Ghi chú: có `n` công nhân và `m` xe đạp (`n <= m`) trên mặt phẳng. Lặp lại: trong các cặp (công nhân, xe) **còn rảnh**, chọn cặp có **khoảng cách Manhattan nhỏ nhất**; nếu hòa thì chọn **chỉ số công nhân nhỏ nhất**; nếu vẫn hòa thì chọn **chỉ số xe nhỏ nhất**. Gán cặp đó rồi lặp lại cho đến khi mọi công nhân có xe. Vì luật hòa đã **xác định duy nhất** kết quả nên đáp án là **duy nhất** (không phải bài nhiều đáp án): ta so sánh trực tiếp. Ý tưởng chính: sắp xếp (hoặc xếp vào các "xô") mọi cặp theo `(khoảng cách, công nhân, xe)` rồi duyệt theo thứ tự đó, cặp nào có cả công nhân lẫn xe còn rảnh thì gán.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: một quá trình **chọn cặp tốt nhất từng bước** với luật phá hòa `(khoảng cách, chỉ số công nhân, chỉ số xe)`.
2. Nắm các công cụ Python cần thiết: khoảng cách Manhattan `abs(dx) + abs(dy)`, so sánh và sắp xếp tuple, `sorted` ổn định, `heapq`, danh sách các "xô" (bucket), mã hóa cặp `(i, j)` thành một số nguyên bằng `i * m + j` và giải mã bằng `divmod`.
3. Hiểu vì sao **duyệt mọi cặp theo thứ tự tăng dần của khóa `(dist, i, j)`** mô phỏng đúng quá trình: cặp đầu tiên có cả hai phía còn rảnh chính là cặp tốt nhất lúc đó.
4. Đi qua 4 cách: mô phỏng trực tiếp (brute force, dùng làm oracle), sắp xếp mọi cặp, xếp xô theo khoảng cách (cách nộp chính, `O(n * m)`) và heap với một ứng viên cho mỗi công nhân.
5. Tránh các lỗi kinh điển: cho từng công nhân tự lấy xe gần nhất theo thứ tự chỉ số, phá hòa sai thứ tự (xe trước công nhân), dùng khoảng cách Euclid, quên đánh dấu công nhân đã có xe, và mảng xô quá nhỏ (khoảng cách Manhattan lớn nhất là `1998`, không phải `1000`).

## 2. Đề bài & ví dụ

Trên sân trường (mặt phẳng X-Y) có `n` công nhân và `m` xe đạp, `n <= m`. Cho mảng `workers` độ dài `n`, `workers[i] = [x_i, y_i]` là vị trí công nhân `i`; mảng `bikes` độ dài `m`, `bikes[j] = [x_j, y_j]` là vị trí xe `j`. Mọi vị trí đều **khác nhau**.

Hãy gán cho mỗi công nhân một chiếc xe. Trong các công nhân và xe còn rảnh, chọn cặp `(worker_i, bike_j)` có **khoảng cách Manhattan** nhỏ nhất và gán. Nếu có nhiều cặp cùng khoảng cách nhỏ nhất thì chọn cặp có **chỉ số công nhân nhỏ nhất**; nếu vẫn còn nhiều cách thì chọn cặp có **chỉ số xe nhỏ nhất**. Lặp lại đến khi mọi công nhân có xe.

Trả về mảng `answer` độ dài `n`, `answer[i]` là chỉ số (tính từ `0`) của xe được gán cho công nhân `i`.

Khoảng cách Manhattan giữa `p1` và `p2` là `|p1.x - p2.x| + |p1.y - p2.y|`.

**Ví dụ 1:** `workers = [[0,0],[2,1]]`, `bikes = [[1,2],[3,3]]` -> `[1,0]`. Công nhân 1 lấy xe 0 vì gần nhất (không hòa), công nhân 0 nhận xe 1.

**Ví dụ 2:** `workers = [[0,0],[1,1],[2,0]]`, `bikes = [[1,0],[2,2],[2,1]]` -> `[0,2,1]`. Công nhân 0 lấy xe 0 trước. Công nhân 1 và 2 cùng cách xe 2 một khoảng như nhau, nên công nhân 1 (chỉ số nhỏ hơn) được xe 2, công nhân 2 nhận xe 1.

**Ràng buộc:**

- `n == workers.length`, `m == bikes.length`
- `1 <= n <= m <= 1000`
- `workers[i].length == bikes[j].length == 2`
- `0 <= x_i, y_i, x_j, y_j < 1000`
- Mọi vị trí công nhân và xe đều khác nhau.

In [1]:
from typing import List
import heapq, random, time
from itertools import permutations

def manhattan(p, q):
    return abs(p[0] - q[0]) + abs(p[1] - q[1])

examples = [
    ([[0, 0], [2, 1]], [[1, 2], [3, 3]], [1, 0]),
    ([[0, 0], [1, 1], [2, 0]], [[1, 0], [2, 2], [2, 1]], [0, 2, 1]),
]
for workers, bikes, expected in examples:
    print(f"workers={workers} bikes={bikes} -> kỳ vọng {expected}")

workers=[[0, 0], [2, 1]] bikes=[[1, 2], [3, 3]] -> kỳ vọng [1, 0]
workers=[[0, 0], [1, 1], [2, 0]] bikes=[[1, 0], [2, 2], [2, 1]] -> kỳ vọng [0, 2, 1]


## 3. Kiến thức nền

### 3.1. Khoảng cách Manhattan

`|x1 - x2| + |y1 - y2|`: số bước đi ngang/dọc giữa hai điểm. Với tọa độ `0..999`, khoảng cách lớn nhất là `999 + 999 = 1998`.

In [2]:
print(manhattan((0, 0), (1, 2)), manhattan((2, 1), (3, 3)), manhattan((0, 0), (999, 999)))

3 3 1998


### 3.2. So sánh tuple và sắp xếp ổn định

Python so sánh tuple **từ trái sang phải**: `(1, 5, 7) < (1, 6, 0)`. Vì vậy `sorted` các tuple `(dist, i, j)` cho đúng luật phá hòa của đề. `sorted` còn **ổn định** (stable): các phần tử bằng nhau theo khóa giữ nguyên thứ tự ban đầu.

In [3]:
pairs = [(2, 1, 0), (1, 2, 0), (1, 0, 5), (1, 0, 3)]
print(sorted(pairs))
words = ["bb", "a", "cc", "d"]
print(sorted(words, key=len))             # ổn định: "bb" trước "cc", "a" trước "d"

[(1, 0, 3), (1, 0, 5), (1, 2, 0), (2, 1, 0)]
['a', 'd', 'bb', 'cc']


### 3.3. Xô (bucket) và mã hóa cặp

Khi khóa chính là số nguyên nhỏ (khoảng cách `0..1998`) ta không cần `sort`: tạo `buckets[d]` rồi thêm từng cặp vào xô của khoảng cách đó. Nếu thêm cặp theo thứ tự `i` tăng, rồi `j` tăng thì trong mỗi xô các cặp **đã** theo thứ tự `(i, j)`. Một cặp `(i, j)` có thể mã hóa thành `i * m + j` và giải mã bằng `divmod(code, m)`.

In [4]:
m = 5
code_ = 3 * m + 2
print(code_, divmod(code_, m))
buckets = [[] for _ in range(4)]
for i in range(2):
    for j in range(3):
        buckets[(i + j) % 4].append((i, j))
print(buckets)

17 (3, 2)
[[(0, 0)], [(0, 1), (1, 0)], [(0, 2), (1, 1)], [(1, 2)]]


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Ví dụ 2: `workers = [[0,0],[1,1],[2,0]]`, `bikes = [[1,0],[2,2],[2,1]]`.

Bảng khoảng cách (hàng = công nhân, cột = xe):

| | xe 0 `(1,0)` | xe 1 `(2,2)` | xe 2 `(2,1)` |
|---|---|---|---|
| công nhân 0 `(0,0)` | 1 | 4 | 3 |
| công nhân 1 `(1,1)` | 1 | 2 | 1 |
| công nhân 2 `(2,0)` | 1 | 2 | 1 |

- Khoảng cách nhỏ nhất là `1`, có các cặp `(0,0)`, `(1,0)`, `(1,2)`, `(2,0)`, `(2,2)`. Phá hòa: chỉ số công nhân nhỏ nhất `0`, cặp `(0,0)`: công nhân 0 lấy xe 0.
- Còn công nhân 1, 2 và xe 1, 2. Khoảng cách nhỏ nhất là `1`: các cặp `(1,2)` và `(2,2)`. Chọn công nhân nhỏ nhất `1`: công nhân 1 lấy xe 2.
- Còn công nhân 2 và xe 1: công nhân 2 lấy xe 1. Kết quả `[0, 2, 1]` ✓.

Nhận xét: nếu liệt kê **mọi** cặp theo thứ tự `(dist, i, j)` tăng dần và duyệt, cặp nào có cả hai phía còn rảnh thì gán ngay, ta được đúng quá trình trên.

## 5. Cách 1 — Brute Force: mô phỏng đúng định nghĩa

**Ý tưởng.** Lặp `n` lần: trong **mọi** cặp `(i, j)` mà công nhân `i` chưa có xe và xe `j` còn rảnh, tìm cặp có khóa `(dist, i, j)` nhỏ nhất rồi gán. Đúng nguyên văn đề, không dùng bất kỳ phân tích nào nên là **oracle độc lập**.

In [5]:
def assign_brute(workers: List[List[int]], bikes: List[List[int]]) -> List[int]:
    n, m = len(workers), len(bikes)
    ans = [-1] * n
    used = [False] * m
    for _ in range(n):
        best = None
        for i in range(n):
            if ans[i] != -1:
                continue
            for j in range(m):
                if used[j]:
                    continue
                key = (manhattan(workers[i], bikes[j]), i, j)
                if best is None or key < best:
                    best = key
        _, i, j = best
        ans[i] = j
        used[j] = True
    return ans

for workers, bikes, expected in examples:
    got = assign_brute(workers, bikes)
    print(f"brute = {got}", "OK" if got == expected else "SAI")
    assert got == expected

brute = [1, 0] OK
brute = [0, 2, 1] OK


**Độ phức tạp và nút thắt.** Mỗi lần chọn quét `O(n * m)` cặp, lặp `n` lần ⇒ `O(n^2 * m)`; với `n = m = 1000` là `10^9`. Đo thử:

In [6]:
def random_instance(rng, n, m, grid=1000):
    cells = rng.sample(range(grid * grid), n + m)
    pts = [[c // grid, c % grid] for c in cells]
    return pts[:n], pts[n:]

rng = random.Random(1057)
for n in (20, 40, 80):
    workers, bikes = random_instance(rng, n, n)
    t0 = time.perf_counter()
    assign_brute(workers, bikes)
    dt = time.perf_counter() - t0
    print(f"n=m={n:3}: {dt * 1000:8.1f} ms -> ngoại suy n=m=1000: ~{dt * (1000 / n) ** 3:9.0f} s")

n=m= 20:      1.1 ms -> ngoại suy n=m=1000: ~      139 s
n=m= 40:      7.9 ms -> ngoại suy n=m=1000: ~      124 s
n=m= 80:     64.6 ms -> ngoại suy n=m=1000: ~      126 s


Nút thắt: mỗi vòng ta **quét lại** mọi cặp để tìm cặp nhỏ nhất. Nhưng thứ tự ưu tiên giữa các cặp **không đổi** theo thời gian (chỉ có các cặp bị loại dần). Vậy chỉ cần sắp xếp các cặp **một lần**.

## 6. Ý tưởng thuật toán / bước đột phá

**Khóa cố định.** Mỗi cặp `(i, j)` có khóa `(dist(i, j), i, j)` không đổi trong suốt quá trình. Cặp được chọn ở mỗi bước là cặp có khóa **nhỏ nhất** trong các cặp còn "sống" (cả công nhân và xe còn rảnh). Nếu ta duyệt mọi cặp theo khóa tăng dần, thì cặp đầu tiên gặp mà còn sống chính là cặp được chọn; các cặp đã chết (công nhân hoặc xe đã bị lấy) cứ bỏ qua.

**Khoảng cách chỉ có `1999` giá trị** (`0..1998`), nên thay vì sắp xếp ta xếp vào `1999` xô; thêm cặp theo thứ tự `i` rồi `j` tăng dần thì mỗi xô đã đúng thứ tự phá hòa. Thời gian `O(n * m)`.

| Cách | Ý tưởng |
|---|---|
| 2 | Sắp xếp `n * m` khóa `(dist, i, j)` rồi duyệt |
| 3 | Xếp xô theo khoảng cách, duyệt xô từ nhỏ đến lớn (nộp) |
| 4 | Heap: mỗi công nhân có danh sách xe đã sắp theo `(dist, j)`, heap chứa ứng viên hiện tại của mọi công nhân |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Sắp xếp mọi cặp

**Giải thích từng đoạn.**

- Sinh `(dist, i, j)` cho mọi cặp.
- `sorted` theo khóa đó (tuple so sánh từ trái sang phải đúng luật đề).
- Duyệt: nếu công nhân `i` chưa có xe và xe `j` chưa dùng thì gán; dừng khi đủ `n` công nhân.

In [7]:
def assign_sorted(workers: List[List[int]], bikes: List[List[int]]) -> List[int]:
    n, m = len(workers), len(bikes)
    pairs = sorted((manhattan(workers[i], bikes[j]), i, j) for i in range(n) for j in range(m))
    ans = [-1] * n
    used = [False] * m
    done = 0
    for _, i, j in pairs:
        if ans[i] == -1 and not used[j]:
            ans[i] = j
            used[j] = True
            done += 1
            if done == n:
                break
    return ans

for workers, bikes, expected in examples:
    got = assign_sorted(workers, bikes)
    print(f"sắp xếp = {got}", "OK" if got == expected else "SAI")
    assert got == expected

sắp xếp = [1, 0] OK
sắp xếp = [0, 2, 1] OK


**Code nộp LeetCode (Cách 2):** mã hóa mỗi cặp thành một số nguyên `dist * (n * m) + i * m + j` để sắp xếp nhanh hơn tuple.

In [8]:
class Solution:
    def assignBikes(self, workers: List[List[int]], bikes: List[List[int]]) -> List[int]:
        n, m = len(workers), len(bikes)
        keys = []
        for i, (wx, wy) in enumerate(workers):
            for j, (bx, by) in enumerate(bikes):
                d = abs(wx - bx) + abs(wy - by)
                keys.append(d * n * m + i * m + j)        # (dist, i, j) mã hóa thành một số nguyên
        keys.sort()
        ans = [-1] * n
        used = [False] * m
        done = 0
        for key in keys:
            ij = key % (n * m)
            i, j = divmod(ij, m)
            if ans[i] == -1 and not used[j]:
                ans[i] = j
                used[j] = True
                done += 1
                if done == n:
                    break
        return ans

sol_sorted = lambda w, b: Solution().assignBikes(w, b)
for workers, bikes, expected in examples:
    assert sol_sorted(workers, bikes) == expected
print("Cách 2 qua ví dụ")

Cách 2 qua ví dụ


**Độ phức tạp (Cách 2):** thời gian `O(n * m * log(n * m))`, bộ nhớ `O(n * m)`.

---

### 7.2. Cách 3 — Xếp xô theo khoảng cách (cách nộp chính)

**Giải thích từng đoạn.**

- `buckets = [[] for _ in range(1999)]`: xô `d` chứa các cặp có khoảng cách `d`. Khoảng cách Manhattan lớn nhất là `999 + 999 = 1998`.
- Duyệt `i` tăng dần, `j` tăng dần và thêm `(i, j)` vào xô của khoảng cách tương ứng. Vì thêm theo thứ tự `(i, j)` nên trong mỗi xô các cặp đã sắp đúng luật phá hòa.
- Duyệt xô `d = 0, 1, 2, ...` và trong xô theo thứ tự; cặp nào còn cả công nhân lẫn xe rảnh thì gán.
- Dừng khi gán đủ `n` công nhân.

In [9]:
def assign_bucket(workers: List[List[int]], bikes: List[List[int]]) -> List[int]:
    n, m = len(workers), len(bikes)
    buckets = [[] for _ in range(1999)]
    for i, (wx, wy) in enumerate(workers):
        for j, (bx, by) in enumerate(bikes):
            buckets[abs(wx - bx) + abs(wy - by)].append(i * m + j)
    ans = [-1] * n
    used = [False] * m
    done = 0
    for bucket in buckets:
        for code_ in bucket:
            i, j = divmod(code_, m)
            if ans[i] == -1 and not used[j]:
                ans[i] = j
                used[j] = True
                done += 1
                if done == n:
                    return ans
    return ans

for workers, bikes, expected in examples:
    got = assign_bucket(workers, bikes)
    print(f"xô = {got}", "OK" if got == expected else "SAI")
    assert got == expected

xô = [1, 0] OK
xô = [0, 2, 1] OK


**Code nộp LeetCode (Cách 3):**

In [10]:
class Solution:
    def assignBikes(self, workers: List[List[int]], bikes: List[List[int]]) -> List[int]:
        n, m = len(workers), len(bikes)
        buckets = [[] for _ in range(1999)]               # khoảng cách Manhattan nằm trong 0..1998
        for i, (wx, wy) in enumerate(workers):
            for j, (bx, by) in enumerate(bikes):
                buckets[abs(wx - bx) + abs(wy - by)].append(i * m + j)   # thêm theo thứ tự (i, j): đã đúng luật phá hòa
        ans = [-1] * n
        used = [False] * m                                  # xe đã bị lấy
        done = 0
        for bucket in buckets:                              # khoảng cách tăng dần
            for code in bucket:
                i, j = divmod(code, m)
                if ans[i] == -1 and not used[j]:            # cả công nhân và xe còn rảnh
                    ans[i] = j
                    used[j] = True
                    done += 1
                    if done == n:
                        return ans
        return ans

sol_bucket = lambda w, b: Solution().assignBikes(w, b)
for workers, bikes, expected in examples:
    assert sol_bucket(workers, bikes) == expected
print("Cách 3 qua ví dụ")

Cách 3 qua ví dụ


**Độ phức tạp (Cách 3):** thời gian `O(n * m + D)` với `D = 1999` là số xô; bộ nhớ `O(n * m)`.

---

### 7.3. Cách 4 — Heap với một ứng viên cho mỗi công nhân

**Ý tưởng khác.** Mỗi công nhân `i` có danh sách xe sắp theo `(dist, j)`. Heap giữ **một** ứng viên cho mỗi công nhân còn chưa có xe: `(dist, i, j)` là xe tốt nhất chưa bị "loại" của công nhân đó. Lấy phần tử nhỏ nhất của heap: nếu xe `j` còn rảnh thì gán; nếu đã bị lấy thì công nhân `i` chuyển sang ứng viên kế tiếp trong danh sách của mình rồi đẩy vào heap.

Phần tử nhỏ nhất heap là cặp có khóa nhỏ nhất trong **tất cả** các cặp còn sống, vì ứng viên hiện tại của mỗi công nhân là cặp nhỏ nhất của công nhân đó trong các cặp chưa bị loại.

In [11]:
def assign_heap(workers: List[List[int]], bikes: List[List[int]]) -> List[int]:
    n, m = len(workers), len(bikes)
    lists = []                                                     # lists[i]: các (dist, j) của công nhân i, đã sắp
    for wx, wy in workers:
        lists.append(sorted((abs(wx - bx) + abs(wy - by), j) for j, (bx, by) in enumerate(bikes)))
    ptr = [0] * n
    heap = [(lists[i][0][0], i, lists[i][0][1]) for i in range(n)]
    heapq.heapify(heap)
    ans = [-1] * n
    used = [False] * m
    while heap:
        d, i, j = heapq.heappop(heap)
        if not used[j]:
            ans[i] = j
            used[j] = True
        else:
            ptr[i] += 1
            nd, nj = lists[i][ptr[i]]
            heapq.heappush(heap, (nd, i, nj))
    return ans

for workers, bikes, expected in examples:
    got = assign_heap(workers, bikes)
    print(f"heap = {got}", "OK" if got == expected else "SAI")
    assert got == expected

heap = [1, 0] OK
heap = [0, 2, 1] OK


**Code nộp LeetCode (Cách 4):** dùng số nguyên `dist * m + j` cho danh sách của từng công nhân.

In [12]:
class Solution:
    def assignBikes(self, workers: List[List[int]], bikes: List[List[int]]) -> List[int]:
        import heapq
        n, m = len(workers), len(bikes)
        lists = []                                          # lists[i]: dist * m + j của công nhân i, tăng dần
        for wx, wy in workers:
            lists.append(sorted((abs(wx - bx) + abs(wy - by)) * m + j for j, (bx, by) in enumerate(bikes)))
        ptr = [0] * n                                       # ứng viên hiện tại của mỗi công nhân
        heap = [(lists[i][0] // m, i, lists[i][0] % m) for i in range(n)]   # (dist, công nhân, xe): đúng luật phá hòa
        heapq.heapify(heap)
        ans = [-1] * n
        used = [False] * m
        while heap:
            d, i, j = heapq.heappop(heap)
            if not used[j]:
                ans[i] = j
                used[j] = True
            else:                                           # xe đã bị lấy: công nhân i xét xe kế tiếp
                ptr[i] += 1
                nd, nj = divmod(lists[i][ptr[i]], m)
                heapq.heappush(heap, (nd, i, nj))
        return ans

sol_heap = lambda w, b: Solution().assignBikes(w, b)
for workers, bikes, expected in examples:
    assert sol_heap(workers, bikes) == expected
print("Cách 4 qua ví dụ")

Cách 4 qua ví dụ


**Độ phức tạp (Cách 4):** sắp xếp danh sách của mọi công nhân `O(n * m log m)`; mỗi lần pop hoặc gán hoặc tăng một con trỏ, tối đa `n * m` lần, mỗi lần `O(log n)`; tổng `O(n * m log (n * m))`, bộ nhớ `O(n * m)`.

## 8. TRACE từng bước

### 8.1. Xô cho ví dụ 2

In các xô khác rỗng và từng bước gán.

In [13]:
def trace_bucket(workers, bikes):
    n, m = len(workers), len(bikes)
    buckets = {}
    for i, w in enumerate(workers):
        for j, b in enumerate(bikes):
            buckets.setdefault(manhattan(w, b), []).append((i, j))
    ans, used, done = [-1] * n, [False] * m, 0
    for d in sorted(buckets):
        print(f"xô d={d}: {buckets[d]}")
        for i, j in buckets[d]:
            if ans[i] == -1 and not used[j]:
                ans[i], used[j], done = j, True, done + 1
                print(f"   gán công nhân {i} <- xe {j}  (khoảng cách {d})  ans = {ans}")
            else:
                why = "công nhân đã có xe" if ans[i] != -1 else "xe đã bị lấy"
                print(f"   bỏ qua ({i}, {j}): {why}")
        if done == n:
            print("   đủ công nhân, dừng")
            break
    return ans

assert trace_bucket(*examples[1][:2]) == [0, 2, 1]

xô d=1: [(0, 0), (1, 0), (1, 2), (2, 0), (2, 2)]
   gán công nhân 0 <- xe 0  (khoảng cách 1)  ans = [0, -1, -1]
   bỏ qua (1, 0): xe đã bị lấy
   gán công nhân 1 <- xe 2  (khoảng cách 1)  ans = [0, 2, -1]
   bỏ qua (2, 0): xe đã bị lấy
   bỏ qua (2, 2): xe đã bị lấy
xô d=2: [(1, 1), (2, 1)]
   bỏ qua (1, 1): công nhân đã có xe
   gán công nhân 2 <- xe 1  (khoảng cách 2)  ans = [0, 2, 1]
   đủ công nhân, dừng


### 8.2. Heap cho ví dụ 2

In phần tử pop ra và việc gán hoặc chuyển ứng viên.

In [14]:
def trace_heap(workers, bikes):
    n, m = len(workers), len(bikes)
    lists = [sorted((manhattan(w, b), j) for j, b in enumerate(bikes)) for w in workers]
    print("danh sách của từng công nhân (dist, xe):", lists)
    ptr = [0] * n
    heap = [(lists[i][0][0], i, lists[i][0][1]) for i in range(n)]
    heapq.heapify(heap)
    ans, used = [-1] * n, [False] * m
    while heap:
        d, i, j = heapq.heappop(heap)
        if not used[j]:
            ans[i], used[j] = j, True
            print(f"pop ({d}, {i}, {j}): xe còn rảnh -> gán, ans = {ans}")
        else:
            ptr[i] += 1
            nd, nj = lists[i][ptr[i]]
            heapq.heappush(heap, (nd, i, nj))
            print(f"pop ({d}, {i}, {j}): xe {j} đã bị lấy -> công nhân {i} chuyển sang ({nd}, {i}, {nj})")
    return ans

assert trace_heap(*examples[1][:2]) == [0, 2, 1]

danh sách của từng công nhân (dist, xe): [[(1, 0), (3, 2), (4, 1)], [(1, 0), (1, 2), (2, 1)], [(1, 0), (1, 2), (2, 1)]]
pop (1, 0, 0): xe còn rảnh -> gán, ans = [0, -1, -1]
pop (1, 1, 0): xe 0 đã bị lấy -> công nhân 1 chuyển sang (1, 1, 2)
pop (1, 1, 2): xe còn rảnh -> gán, ans = [0, 2, -1]
pop (1, 2, 0): xe 0 đã bị lấy -> công nhân 2 chuyển sang (1, 2, 2)
pop (1, 2, 2): xe 2 đã bị lấy -> công nhân 2 chuyển sang (2, 2, 1)
pop (2, 2, 1): xe còn rảnh -> gán, ans = [0, 2, 1]


### 8.3. Trường hợp "tham lam theo từng công nhân" cho kết quả khác

`workers = [[0,0],[0,1]]`, `bikes = [[0,2],[0,3]]`. Công nhân 1 (gần xe 0 nhất, khoảng cách `1`) phải được ưu tiên trước công nhân 0 dù chỉ số lớn hơn.

In [15]:
workers, bikes = [[0, 0], [0, 1]], [[0, 2], [0, 3]]
naive = []
free = set(range(len(bikes)))
for w in workers:                                 # mỗi công nhân (theo thứ tự chỉ số) lấy xe gần nhất còn rảnh
    j = min(free, key=lambda k: (manhattan(w, bikes[k]), k))
    naive.append(j)
    free.remove(j)
print("tham lam theo công nhân:", naive, "| đúng:", assign_bucket(workers, bikes))
assert naive == [0, 1] and assign_bucket(workers, bikes) == [1, 0]

tham lam theo công nhân: [0, 1] | đúng: [1, 0]


## 9. Code nộp LeetCode

Ta chọn **Cách 3** (xếp xô): `O(n * m)` thời gian, không cần `sort`.

```python
class Solution:
    def assignBikes(self, workers: List[List[int]], bikes: List[List[int]]) -> List[int]:
        n, m = len(workers), len(bikes)
        buckets = [[] for _ in range(1999)]               # khoảng cách Manhattan nằm trong 0..1998
        for i, (wx, wy) in enumerate(workers):
            for j, (bx, by) in enumerate(bikes):
                buckets[abs(wx - bx) + abs(wy - by)].append(i * m + j)   # thêm theo thứ tự (i, j): đã đúng luật phá hòa
        ans = [-1] * n
        used = [False] * m                                  # xe đã bị lấy
        done = 0
        for bucket in buckets:                              # khoảng cách tăng dần
            for code in bucket:
                i, j = divmod(code, m)
                if ans[i] == -1 and not used[j]:            # cả công nhân và xe còn rảnh
                    ans[i] = j
                    used[j] = True
                    done += 1
                    if done == n:
                        return ans
        return ans
```

In [16]:
class Solution:
    def assignBikes(self, workers: List[List[int]], bikes: List[List[int]]) -> List[int]:
        n, m = len(workers), len(bikes)
        buckets = [[] for _ in range(1999)]               # khoảng cách Manhattan nằm trong 0..1998
        for i, (wx, wy) in enumerate(workers):
            for j, (bx, by) in enumerate(bikes):
                buckets[abs(wx - bx) + abs(wy - by)].append(i * m + j)   # thêm theo thứ tự (i, j): đã đúng luật phá hòa
        ans = [-1] * n
        used = [False] * m                                  # xe đã bị lấy
        done = 0
        for bucket in buckets:                              # khoảng cách tăng dần
            for code in bucket:
                i, j = divmod(code, m)
                if ans[i] == -1 and not used[j]:            # cả công nhân và xe còn rảnh
                    ans[i] = j
                    used[j] = True
                    done += 1
                    if done == n:
                        return ans
        return ans

sol = lambda w, b: Solution().assignBikes(w, b)
for workers, bikes, expected in examples:
    got = sol(workers, bikes)
    print(got, "OK" if got == expected else "SAI")
    assert got == expected

[1, 0] OK
[0, 2, 1] OK


## 10. Vì sao đúng

Gọi khóa của cặp `(i, j)` là `K(i, j) = (dist(i, j), i, j)`. Hai cặp khác nhau luôn có khóa khác nhau (vì khác `i` hoặc khác `j`), nên khóa nhỏ nhất luôn **duy nhất**; vì vậy luật của đề xác định đáp án duy nhất.

**Khẳng định.** Quá trình của đề tại mỗi bước chọn cặp sống có khóa nhỏ nhất, trong đó "sống" nghĩa là công nhân và xe cùng còn rảnh. Thuật toán duyệt mọi cặp theo khóa tăng dần và gán cặp đầu tiên còn sống. Quy nạp theo số bước: giả sử `k` bước đầu hai quá trình chọn giống nhau. Ở bước `k + 1`, mọi cặp có khóa nhỏ hơn cặp `P` mà thuật toán chọn đều đã chết tại thời điểm thuật toán xét đến chúng (nếu chúng còn sống thì đã được chọn trước `P`), và vì công nhân/xe đã bị lấy **không bao giờ được trả lại**, chúng vẫn chết ở bước `k + 1`. Còn `P` thì sống. Vậy `P` là cặp sống có khóa nhỏ nhất, đúng cặp mà đề chọn.

**Xô.** Các cặp được thêm vào xô theo thứ tự `i` tăng, rồi `j` tăng. Trong cùng một xô (cùng `dist`) thứ tự thêm vào chính là thứ tự tăng của `(i, j)`; các xô duyệt theo `dist` tăng. Nên duyệt xô cho đúng thứ tự khóa tăng dần, giống như `sorted`.

**Heap (Cách 4).** Với mỗi công nhân chưa có xe ta giữ cặp nhỏ nhất của công nhân đó trong các cặp có xe chưa bị loại **tính đến lúc xét**. Phần tử nhỏ nhất heap là khóa nhỏ nhất trong các ứng viên đó. Nếu xe của ứng viên đã bị lấy, bỏ nó và đưa ứng viên kế tiếp (khóa lớn hơn, vì danh sách đã sắp). Nếu xe còn rảnh thì ứng viên này là cặp sống nhỏ nhất toàn cục (mọi cặp sống khác có khóa lớn hơn ứng viên hiện tại của công nhân đó, mà ứng viên hiện tại là nhỏ nhất của công nhân đó), do đó gán là đúng.

**Nhận xét thú vị (phá hòa `(dist, i, j)` hay `(dist, j, i)` đều cho cùng kết quả).** Hai cặp chỉ ảnh hưởng nhau khi chúng **chung công nhân hoặc chung xe**. Nếu chung công nhân `i`, cặp có `j` nhỏ hơn thắng ở cả hai thứ tự; nếu chung xe `j`, cặp có `i` nhỏ hơn thắng ở cả hai thứ tự. Hai cặp không chung gì thì thứ tự giữa chúng không ảnh hưởng kết quả. Vì vậy đảo thứ tự ưu tiên `(i, j)` thành `(j, i)` không đổi đáp án; ta kiểm bằng máy ở mục 12.7. Điều **không** được đảo là chiều tăng/giảm của chỉ số.

**Dừng.** Mỗi công nhân có ít nhất một xe (vì `n <= m`) và mỗi lần gán dùng đúng một công nhân và một xe; còn xe rảnh cho mọi công nhân còn lại, nên thuật toán luôn gán đủ `n` công nhân.

**Kiểm chứng bằng máy.** Oracle (Cách 1) mô phỏng đúng định nghĩa. Ta đối chiếu mọi cách với oracle trên rất nhiều ví dụ ngẫu nhiên trên lưới nhỏ (nhiều hòa), cộng thêm một hàm kiểm tra tính hợp lệ độc lập (xem mục 12).

In [17]:
rng = random.Random(10)
total = 0
for _ in range(3000):
    grid = rng.choice([3, 4, 5, 8, 1000])
    cells = grid * grid
    n = rng.randint(1, min(6, cells // 2))
    m = rng.randint(n, min(8, cells - n))
    workers, bikes = random_instance(rng, n, m, grid)
    ref = assign_brute(workers, bikes)
    assert assign_sorted(workers, bikes) == assign_bucket(workers, bikes) == assign_heap(workers, bikes) == ref, (workers, bikes)
    assert sol_sorted(workers, bikes) == sol_bucket(workers, bikes) == sol_heap(workers, bikes) == sol(workers, bikes) == ref
    total += 1
print("đã đối chiếu", total, "ví dụ ngẫu nhiên: mọi cách khớp oracle")

đã đối chiếu 3000 ví dụ ngẫu nhiên: mọi cách khớp oracle


## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Mô phỏng | `O(n^2 * m)` | `O(n + m)` |
| 2. Sắp xếp mọi cặp | `O(n m log(n m))` | `O(n m)` |
| 3. Xếp xô | `O(n m + D)`, `D = 1999` | `O(n m)` |
| 4. Heap | `O(n m log(n m))` | `O(n m)` |

Đo ở cỡ tối đa (`n = m = 1000`, tức `10^6` cặp) với vài dạng đầu vào:

In [18]:
rng = random.Random(11)
cases = {}
cases["ngẫu nhiên n=m=1000"] = random_instance(rng, 1000, 1000)
cases["n=100, m=1000"] = random_instance(rng, 100, 1000)
# nhiều hòa: công nhân ở cột 0..49 (hàng 0..19), xe ở cột 50..99
w = [[x, y] for x in range(50) for y in range(20)]
b = [[50 + x, y] for x in range(50) for y in range(20)]
cases["lưới nhiều hòa n=m=1000"] = (w, b)
# cực đoan: mọi công nhân gần cùng một cụm xe
cases["tất cả công nhân gần một cụm xe"] = ([[x, 0] for x in range(300)], [[x, 999] for x in range(300)] + [[x + 400, 500] for x in range(600)])
for label, (w, b) in cases.items():
    ref = None
    row = [f"{label:34}"]
    for name, fn in (("xô", sol_bucket), ("sắp xếp", sol_sorted), ("heap", sol_heap)):
        t0 = time.perf_counter()
        got = fn(w, b)
        dt = (time.perf_counter() - t0) * 1000
        if ref is None:
            ref = got
        assert got == ref, (label, name)
        row.append(f"{name} {dt:8.1f} ms")
    print(" | ".join(row))

ngẫu nhiên n=m=1000                | xô    469.1 ms | sắp xếp    480.4 ms | heap    520.3 ms
n=100, m=1000                      | xô     21.0 ms | sắp xếp     20.3 ms | heap     19.1 ms


lưới nhiều hòa n=m=1000            | xô    284.5 ms | sắp xếp    301.4 ms | heap    267.1 ms
tất cả công nhân gần một cụm xe    | xô     56.2 ms | sắp xếp     55.0 ms | heap     54.9 ms


## 12. Bộ test

Bài có đáp án **duy nhất** nên ta so sánh trực tiếp với oracle; thêm một **hàm kiểm tra tính hợp lệ độc lập** (`is_valid_assignment`): đáp án phải gán `n` xe **khác nhau**, và khi xếp các cặp được gán theo khóa `(dist, i, j)` thì mỗi cặp phải là cặp nhỏ nhất trong các cặp còn sống tại thời điểm đó. Hàm này không dùng thuật toán nào ở trên. Bộ test gồm: ví dụ của đề, edge case có kỳ vọng viết tay, kiểm tra hàm kiểm tra từ chối kết quả sai, phản ví dụ cho các cách sai (tìm bằng tìm kiếm), test ngẫu nhiên có seed (đúng ràng buộc) và test cỡ lớn.

In [19]:
def is_valid_assignment(workers, bikes, ans) -> bool:
    n, m = len(workers), len(bikes)
    if not isinstance(ans, list) or len(ans) != n or any(not (0 <= j < m) for j in ans) or len(set(ans)) != n:
        return False
    assigned = sorted((manhattan(workers[i], bikes[ans[i]]), i, ans[i]) for i in range(n))
    free_w, free_b = set(range(n)), set(range(m))
    for key in assigned:
        _, i, j = key
        # không được tồn tại cặp sống nào có khóa nhỏ hơn key
        for i2 in free_w:
            for j2 in free_b:
                if (manhattan(workers[i2], bikes[j2]), i2, j2) < key:
                    return False
        free_w.discard(i)
        free_b.discard(j)
    return True

def check_all(workers, bikes, expected=None, brute=True, validate=True):
    results = {name: fn(workers, bikes) for name, fn in (("xô", sol_bucket), ("sắp xếp", sol_sorted), ("heap", sol_heap), ("nộp", sol),
                                                         ("assign_sorted", assign_sorted), ("assign_bucket", assign_bucket), ("assign_heap", assign_heap))}
    first = results["nộp"]
    for name, got in results.items():
        assert got == first, (name, got[:10], first[:10])
    if brute and len(workers) <= 40:
        assert assign_brute(workers, bikes) == first
    if validate and len(workers) <= 12 and len(bikes) <= 14:
        assert is_valid_assignment(workers, bikes, first)
    if expected is not None:
        assert first == expected, (first, expected)
    return first

# 12.1 ví dụ đề
for w, b, e in examples:
    check_all(w, b, e)
print("12.1 ví dụ đề: OK")

# 12.2 edge case (kỳ vọng viết tay, kèm lý do)
edge = [
    ([[0, 0]], [[1, 1]], [0]),                                         # một công nhân, một xe
    ([[0, 0]], [[5, 5], [1, 0], [0, 1]], [1]),                         # n = 1: xe 1 và 2 cùng khoảng cách 1, chọn xe chỉ số nhỏ
    ([[0, 0], [2, 0]], [[1, 0], [5, 5]], [0, 1]),                      # hai công nhân cùng cách xe 0: công nhân 0 trước
    ([[0, 0], [0, 1]], [[0, 2], [0, 3]], [1, 0]),                      # công nhân 1 gần xe 0 hơn nên được xe 0 trước
    ([[0, 0]], [[999, 999]], [0]),                                     # khoảng cách lớn nhất 1998
    ([[0, 0], [0, 2]], [[0, 1], [0, 3]], [0, 1]),                      # hòa khoảng cách 1: (0,0) trước (1,0), rồi (1,1)
    ([[0, 0], [999, 999]], [[1, 0], [998, 999]], [0, 1]),              # hai cụm tách biệt
    ([[1, 1], [0, 0]], [[0, 1], [1, 0]], [0, 1]),                      # mọi khoảng cách bằng 1: công nhân 0 lấy xe 0, công nhân 1 lấy xe 1
]
for w, b, e in edge:
    check_all(w, b, e)
print("12.2 edge case: OK", len(edge), "ca")

# 12.3 hàm kiểm tra phải từ chối kết quả sai
w, b = [[0, 0], [0, 1]], [[0, 2], [0, 3]]
assert is_valid_assignment(w, b, [1, 0])
assert not is_valid_assignment(w, b, [0, 1])          # gán theo từng công nhân: sai
assert not is_valid_assignment(w, b, [1, 1])          # hai công nhân cùng một xe
assert not is_valid_assignment(w, b, [1])             # thiếu công nhân
assert not is_valid_assignment(w, b, [1, 2])          # xe không tồn tại
print("12.3 hàm kiểm tra: OK")

12.1 ví dụ đề: OK
12.2 edge case: OK 8 ca
12.3 hàm kiểm tra: OK


In [20]:
# 12.4 phản ví dụ: các cách sai (tìm bằng tìm kiếm so với oracle)
def wrong_per_worker(workers, bikes):                 # mỗi công nhân (theo chỉ số) lấy xe gần nhất còn rảnh
    free = set(range(len(bikes)))
    ans = []
    for w in workers:
        j = min(free, key=lambda k: (manhattan(w, bikes[k]), k))
        ans.append(j)
        free.remove(j)
    return ans

def wrong_tie_reversed(workers, bikes):               # phá hòa ngược: chỉ số công nhân LỚN nhất trước, rồi chỉ số xe nhỏ nhất
    n, m = len(workers), len(bikes)
    pairs = sorted((manhattan(workers[i], bikes[j]), -i, j) for i in range(n) for j in range(m))
    ans, used = [-1] * n, [False] * m
    for _, ni, j in pairs:
        i = -ni
        if ans[i] == -1 and not used[j]:
            ans[i], used[j] = j, True
    return ans

def wrong_euclid(workers, bikes):                     # dùng bình phương khoảng cách Euclid thay cho Manhattan
    n, m = len(workers), len(bikes)
    pairs = sorted(((workers[i][0] - bikes[j][0]) ** 2 + (workers[i][1] - bikes[j][1]) ** 2, i, j) for i in range(n) for j in range(m))
    ans, used = [-1] * n, [False] * m
    for _, i, j in pairs:
        if ans[i] == -1 and not used[j]:
            ans[i], used[j] = j, True
    return ans

def wrong_no_worker_flag(workers, bikes):             # quên đánh dấu công nhân đã có xe
    n, m = len(workers), len(bikes)
    pairs = sorted((manhattan(workers[i], bikes[j]), i, j) for i in range(n) for j in range(m))
    ans, used = [-1] * n, [False] * m
    for _, i, j in pairs:
        if not used[j]:
            ans[i], used[j] = j, True
    return ans

def tie_swapped(workers, bikes):                      # phá hòa theo (dist, xe, công nhân): KHÔNG sai, xem mục 10
    n, m = len(workers), len(bikes)
    pairs = sorted((manhattan(workers[i], bikes[j]), j, i) for i in range(n) for j in range(m))
    ans, used = [-1] * n, [False] * m
    for _, j, i in pairs:
        if ans[i] == -1 and not used[j]:
            ans[i], used[j] = j, True
    return ans

def find_counter(wrong, tries=60000, seed=9):
    rng = random.Random(seed)
    for _ in range(tries):
        grid = rng.choice([3, 4, 5])
        n = rng.randint(2, 4)
        m = rng.randint(n, 5)
        if n + m > grid * grid:
            continue
        w, b = random_instance(rng, n, m, grid)
        if wrong(w, b) != assign_brute(w, b):
            return w, b
    return None

counters = {}
for name, fn in (("mỗi công nhân lấy xe gần nhất theo thứ tự chỉ số", wrong_per_worker), ("phá hòa ngược (công nhân lớn nhất trước)", wrong_tie_reversed),
                 ("dùng khoảng cách Euclid", wrong_euclid), ("quên đánh dấu công nhân đã có xe", wrong_no_worker_flag)):
    case = find_counter(fn)
    assert case is not None, name
    counters[name] = case
    w, b = case
    print(f"{name:52} workers={w} bikes={b}: sai -> {fn(w, b)}, đúng -> {assign_brute(w, b)}")
    check_all(w, b)
print("12.4 phản ví dụ: OK")

mỗi công nhân lấy xe gần nhất theo thứ tự chỉ số     workers=[[2, 1], [0, 3], [2, 2], [3, 2]] bikes=[[2, 3], [1, 2], [0, 1], [1, 0]]: sai -> [0, 1, 2, 3], đúng -> [1, 2, 0, 3]
phá hòa ngược (công nhân lớn nhất trước)             workers=[[3, 0], [0, 2], [2, 3], [1, 3]] bikes=[[3, 1], [1, 2], [3, 2], [2, 1]]: sai -> [0, 3, 2, 1], đúng -> [0, 1, 2, 3]
dùng khoảng cách Euclid                              workers=[[2, 0], [0, 2], [3, 2], [2, 2]] bikes=[[0, 0], [1, 1], [3, 3], [1, 3], [0, 1]]: sai -> [1, 4, 2, 3], đúng -> [0, 4, 2, 1]
quên đánh dấu công nhân đã có xe                     workers=[[2, 0], [0, 2], [3, 2], [2, 2]] bikes=[[0, 0], [1, 1], [3, 3], [1, 3], [0, 1]]: sai -> [1, 3, 2, -1], đúng -> [0, 4, 2, 1]
12.4 phản ví dụ: OK


In [21]:
# 12.5 ngẫu nhiên nhỏ: lưới nhỏ (nhiều hòa) và lưới thật
rng = random.Random(4057)
for _ in range(3000):
    grid = rng.choice([3, 4, 5, 10, 1000])
    n = rng.randint(1, min(7, grid * grid // 2))
    m = rng.randint(n, min(9, grid * grid - n))
    w, b = random_instance(rng, n, m, grid)
    check_all(w, b, validate=n <= 6 and m <= 8)
print("12.5 ngẫu nhiên nhỏ: OK")

# 12.6 ngẫu nhiên vừa (n <= 40, so oracle) và lớn (so các cách nhanh với nhau)
for _ in range(60):
    grid = rng.choice([12, 40, 1000])
    n = rng.randint(10, 40)
    m = rng.randint(n, 60)
    w, b = random_instance(rng, n, m, grid)
    check_all(w, b, validate=False)
for _ in range(4):
    grid = rng.choice([40, 100, 1000])
    n = rng.randint(200, 400)
    m = rng.randint(n, 500)
    w, b = random_instance(rng, n, m, grid)
    check_all(w, b, brute=False, validate=False)
print("12.6 ngẫu nhiên vừa và lớn: OK")

# 12.7 tính chất: kết quả là đơn ánh; nếu thêm xe rất xa thì đáp án không đổi
for _ in range(300):
    n = rng.randint(1, 6)
    m = rng.randint(n, 8)
    w, b = random_instance(rng, n, m, 50)
    ans = sol(w, b)
    assert len(set(ans)) == n and all(0 <= j < m for j in ans)
    far = [[999, 999]]                                              # xe thêm vào cuối, xa mọi công nhân trong lưới 50 x 50
    assert sol(w, b + far) == ans
for _ in range(500):                                                 # phá hòa (dist, j, i) cho cùng đáp án với (dist, i, j)
    grid = rng.choice([3, 4, 5, 10])
    n = rng.randint(1, min(6, grid * grid // 2))
    m = rng.randint(n, min(8, grid * grid - n))
    w, b = random_instance(rng, n, m, grid)
    assert tie_swapped(w, b) == assign_brute(w, b), (w, b)
print("12.7 tính chất: OK")

12.5 ngẫu nhiên nhỏ: OK


12.6 ngẫu nhiên vừa và lớn: OK
12.7 tính chất: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Cho từng công nhân (theo thứ tự chỉ số) lấy xe gần nhất

Đề yêu cầu chọn **cặp tốt nhất trên toàn cục** trước, không phải đi theo thứ tự công nhân.

In [22]:
w, b = counters["mỗi công nhân lấy xe gần nhất theo thứ tự chỉ số"]
print("workers =", w, "bikes =", b, "| sai ->", wrong_per_worker(w, b), "| đúng ->", sol(w, b))
assert wrong_per_worker(w, b) != sol(w, b)

workers = [[2, 1], [0, 3], [2, 2], [3, 2]] bikes = [[2, 3], [1, 2], [0, 1], [1, 0]] | sai -> [0, 1, 2, 3] | đúng -> [1, 2, 0, 3]


Vì sao sai: công nhân có chỉ số lớn nhưng đứng sát một chiếc xe phải được ưu tiên hơn công nhân chỉ số nhỏ ở xa. **Đúng:** xét mọi cặp theo khóa `(dist, i, j)`.

### Lỗi 2 — Phá hòa ngược chiều

Đề ưu tiên **chỉ số công nhân nhỏ nhất**. Nếu ghi `-i` vào khóa (hoặc quên rằng ưu tiên là "nhỏ nhất") thì kết quả sai.

In [23]:
w, b = counters["phá hòa ngược (công nhân lớn nhất trước)"]
print("workers =", w, "bikes =", b, "| sai ->", wrong_tie_reversed(w, b), "| đúng ->", sol(w, b))
assert wrong_tie_reversed(w, b) != sol(w, b)

workers = [[3, 0], [0, 2], [2, 3], [1, 3]] bikes = [[3, 1], [1, 2], [3, 2], [2, 1]] | sai -> [0, 3, 2, 1] | đúng -> [0, 1, 2, 3]


Vì sao sai: hai cặp cùng khoảng cách cạnh tranh một công nhân hoặc một xe thì chỉ số nhỏ hơn phải thắng. **Đúng:** khóa `(dist, i, j)` tăng dần. (Điều thú vị: đổi thành `(dist, j, i)` thì **vẫn đúng**, xem mục 10.)

### Lỗi 3 — Dùng khoảng cách Euclid thay cho Manhattan

In [24]:
w, b = counters["dùng khoảng cách Euclid"]
print("workers =", w, "bikes =", b, "| sai ->", wrong_euclid(w, b), "| đúng ->", sol(w, b))
assert wrong_euclid(w, b) != sol(w, b)

workers = [[2, 0], [0, 2], [3, 2], [2, 2]] bikes = [[0, 0], [1, 1], [3, 3], [1, 3], [0, 1]] | sai -> [1, 4, 2, 3] | đúng -> [0, 4, 2, 1]


Vì sao sai: thứ tự theo khoảng cách Euclid khác thứ tự theo Manhattan (ví dụ `(1, 1)` có Manhattan `2` nhưng Euclid bình phương `2`, còn `(0, 2)` có Manhattan `2` nhưng Euclid bình phương `4`). **Đúng:** `abs(dx) + abs(dy)`.

### Lỗi 4 — Quên đánh dấu công nhân đã có xe

Chỉ kiểm tra xe còn rảnh nhưng không kiểm tra công nhân, nên một công nhân có thể bị ghi đè nhiều lần và công nhân khác không có xe.

In [25]:
w, b = counters["quên đánh dấu công nhân đã có xe"]
bad = wrong_no_worker_flag(w, b)
print("workers =", w, "bikes =", b, "| sai ->", bad, "| đúng ->", sol(w, b))
assert bad != sol(w, b)

workers = [[2, 0], [0, 2], [3, 2], [2, 2]] bikes = [[0, 0], [1, 1], [3, 3], [1, 3], [0, 1]] | sai -> [1, 3, 2, -1] | đúng -> [0, 4, 2, 1]


Vì sao sai: mỗi công nhân chỉ được nhận một xe. **Đúng:** điều kiện `ans[i] == -1 and not used[j]`.

### Lỗi 5 — Mảng xô quá nhỏ

Khoảng cách Manhattan lớn nhất là `999 + 999 = 1998`, không phải `999` hay `1000`. Tạo `1000` xô sẽ gây `IndexError` với các cặp xa nhau.

In [26]:
def wrong_small_buckets(workers, bikes):
    n, m = len(workers), len(bikes)
    buckets = [[] for _ in range(1000)]               # SAI: thiếu xô cho khoảng cách 1000..1998
    for i, (wx, wy) in enumerate(workers):
        for j, (bx, by) in enumerate(bikes):
            buckets[abs(wx - bx) + abs(wy - by)].append(i * m + j)
    return buckets

try:
    wrong_small_buckets([[0, 0]], [[999, 999]])
    print("không có lỗi (không mong đợi)")
except IndexError as e:
    print("IndexError:", e)
assert sol([[0, 0]], [[999, 999]]) == [0]

IndexError: list index out of range


Vì sao sai: chỉ số xô là khoảng cách, nên cần `1999` xô. **Đúng:** `[[] for _ in range(1999)]` (hoặc `range(max_dist + 1)`).

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "lặp lại: chọn cặp tốt nhất theo một khóa cố định rồi loại hai phía" ⇒ sắp xếp mọi cặp theo khóa rồi duyệt (tham lam), hoặc dùng heap.
- Khóa nhỏ (khoảng cách `0..1998`) ⇒ **bucket sort** nhanh hơn `sort`.
- Khóa phá hòa ghi **tường minh** trong tuple `(dist, i, j)`, đừng dựa vào thứ tự sinh cặp.
- Mã hóa cặp `(i, j)` thành một số nguyên để tiết kiệm bộ nhớ khi có `10^6` cặp.
- Cùng họ: Campus Bikes II (1066, tối thiểu tổng khoảng cách, bitmask DP), Assign Cookies (455), Stable Marriage.
- Test nên có: `n = 1`, nhiều hòa khoảng cách (lưới nhỏ), khoảng cách lớn nhất `1998`, `n = m = 1000`.

## 15. Biến thể

1. **Campus Bikes II (1066):** gán sao cho **tổng** khoảng cách nhỏ nhất (không còn luật tham lam): quy hoạch động theo tập xe đã dùng (bitmask), `n, m <= 10`.
2. **Thứ tự gán:** trả về danh sách các cặp theo thứ tự được chọn (bài tập 5).
3. Bài liên quan: 1066, 455.

Kiểm biến thể 1 (bitmask DP) với oracle liệt kê mọi hoán vị:

In [27]:
def min_total_distance(workers, bikes):
    n, m = len(workers), len(bikes)
    INF = float("inf")
    dp = [INF] * (1 << m)                              # dp[mask]: tổng nhỏ nhất khi các xe trong mask đã được gán cho công nhân 0..popcount-1
    dp[0] = 0
    for mask in range(1 << m):
        if dp[mask] == INF:
            continue
        i = bin(mask).count("1")
        if i == n:
            continue
        for j in range(m):
            if not mask >> j & 1:
                nm = mask | (1 << j)
                dp[nm] = min(dp[nm], dp[mask] + manhattan(workers[i], bikes[j]))
    return min(dp[mask] for mask in range(1 << m) if bin(mask).count("1") == n)

def min_total_distance_brute(workers, bikes):
    n, m = len(workers), len(bikes)
    return min(sum(manhattan(workers[i], bikes[p[i]]) for i in range(n)) for p in permutations(range(m), n))

rng = random.Random(15)
for _ in range(300):
    n = rng.randint(1, 4)
    m = rng.randint(n, 6)
    w, b = random_instance(rng, n, m, 8)
    assert min_total_distance(w, b) == min_total_distance_brute(w, b), (w, b)
print("biến thể Campus Bikes II: bitmask DP khớp oracle hoán vị")
# Luật tham lam của bài này có thể cho tổng KHÔNG nhỏ nhất: tìm ví dụ bằng tìm kiếm
rng = random.Random(151)
for _ in range(20000):
    w, b = random_instance(rng, 2, 3, 6)
    greedy_total = sum(manhattan(w[i], b[j]) for i, j in enumerate(sol(w, b)))
    if greedy_total > min_total_distance(w, b):
        break
print("workers =", w, "bikes =", b, "| gán theo luật tham lam:", sol(w, b), "tổng", greedy_total, "| tổng nhỏ nhất:", min_total_distance(w, b))
assert greedy_total > min_total_distance(w, b)

biến thể Campus Bikes II: bitmask DP khớp oracle hoán vị
workers = [[4, 1], [3, 0]] bikes = [[1, 5], [3, 5], [5, 5]] | gán theo luật tham lam: [1, 0] tổng 12 | tổng nhỏ nhất: 10


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng | Quét mọi cặp mỗi lần | `O(n^2 m)` | `O(n + m)` | Oracle |
| 2. Sắp xếp mọi cặp | Khóa `(dist, i, j)` | `O(n m log(n m))` | `O(n m)` | Dễ hiểu |
| 3. Xếp xô | Khoảng cách `0..1998` | `O(n m)` | `O(n m)` | **Cách nộp chính** |
| 4. Heap | Một ứng viên mỗi công nhân | `O(n m log(n m))` | `O(n m)` | Khi `m` lớn nhưng ít cặp bị loại |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán rồi chạy để kiểm tra:

```python
print(manhattan((0, 0), (2, 3)), manhattan((5, 1), (2, 5)))
print(sorted([(2, 1, 0), (1, 2, 5), (1, 0, 9), (1, 0, 3)]))
print(Solution().assignBikes([[0, 0]], [[3, 3], [1, 1]]))
```

In [28]:
# Bài 1: ghi dự đoán của bạn
my_guess_1 = None      # hai khoảng cách
my_guess_2 = None      # danh sách đã sắp
my_guess_3 = None      # kết quả assignBikes

### Bài 2 — Tính tay (dễ-vừa)

`workers = [[0,0],[3,0],[1,3]]`, `bikes = [[1,0],[3,1],[0,3],[5,5]]`. Lập bảng khoảng cách, chọn từng cặp theo luật và ghi `answer`.

In [29]:
# Bài 2: điền kết quả tính tay
hand_answer = None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành xếp xô.

In [30]:
def exercise_3(workers, bikes):
    n, m = len(workers), len(bikes)
    buckets = [[] for _ in range(1999)]
    for i, (wx, wy) in enumerate(workers):
        for j, (bx, by) in enumerate(bikes):
            pass
            # TODO 1: thêm i * m + j vào buckets[khoảng cách Manhattan]
    ans = [-1] * n
    used = [False] * m
    for bucket in buckets:
        for code_ in bucket:
            i, j = divmod(code_, m)
            # TODO 2: nếu công nhân i chưa có xe và xe j chưa dùng thì gán
            pass
    return ans


# for w, b, e in examples:
#     assert exercise_3(w, b) == e

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 2, tự viết `exercise_4(workers, bikes)` bằng **sắp xếp** các khóa `(dist, i, j)`.

In [31]:
def exercise_4(workers, bikes):
    # TODO
    pass


# for w, b, e in examples:
#     assert exercise_4(w, b) == e

### Bài 5 — Mini challenge: thứ tự gán (khó)

Viết `exercise_5(workers, bikes)` trả về danh sách các cặp `(i, j)` **theo đúng thứ tự được chọn** trong quá trình của đề. Ví dụ với ví dụ 2 của đề trả về `[(0, 0), (1, 2), (2, 1)]`. Kiểm bằng oracle mô phỏng.

Gợi ý: như xếp xô nhưng ghi lại thứ tự mỗi lần gán.

In [32]:
def exercise_5(workers, bikes):
    # TODO
    pass


def order_oracle(workers, bikes):
    n, m = len(workers), len(bikes)
    ans, used, order = [-1] * n, [False] * m, []
    for _ in range(n):
        best = min((manhattan(workers[i], bikes[j]), i, j) for i in range(n) if ans[i] == -1 for j in range(m) if not used[j])
        _, i, j = best
        ans[i], used[j] = j, True
        order.append((i, j))
    return order


# assert exercise_5(*examples[1][:2]) == [(0, 0), (1, 2), (2, 1)]
# rng = random.Random(5)
# for _ in range(100):
#     w, b = random_instance(rng, 4, 6, 5)
#     assert exercise_5(w, b) == order_oracle(w, b)

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: Manhattan là `|dx| + |dy|`; tuple so sánh từ trái sang phải.
- Mức 2: `(0,0)-(2,3)`: `2 + 3`; `(5,1)-(2,5)`: `3 + 4`. Xe gần hơn là xe `(1,1)`.
- Mức 3: `5 7`; `[(1, 0, 3), (1, 0, 9), (1, 2, 5), (2, 1, 0)]`; `[1]`.

**Bài 2**
- Mức 1: tính bảng khoảng cách `3 x 4`.
- Mức 2: công nhân 0: `[1, 4, 3, 10]`; công nhân 1: `[2, 1, 6, 7]`; công nhân 2: `[3, 4, 1, 6]`. Nhỏ nhất `1`: các cặp `(0,0)`, `(1,1)`, `(2,2)`: gán cả ba theo thứ tự chỉ số vì chúng độc lập.
- Mức 3: `[0, 1, 2]`.

**Bài 3**
- Mức 1: `d = abs(wx - bx) + abs(wy - by)`.
- Mức 2: `buckets[d].append(i * m + j)`.
- Mức 3: `if ans[i] == -1 and not used[j]: ans[i] = j; used[j] = True`.

**Bài 4**
- Mức 1: sinh `(dist, i, j)` cho mọi cặp, `sorted`.
- Mức 2: duyệt, gán khi `ans[i] == -1 and not used[j]`.
- Mức 3: dừng khi đã gán đủ `n`.

**Bài 5**
- Mức 1: giữ danh sách `order`.
- Mức 2: mỗi lần gán thêm `(i, j)` vào `order`.
- Mức 3: trả `order` sau khi duyệt xong.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [33]:
r1 = (manhattan((0, 0), (2, 3)), manhattan((5, 1), (2, 5)))
r2 = sorted([(2, 1, 0), (1, 2, 5), (1, 0, 9), (1, 0, 3)])
r3 = Solution().assignBikes([[0, 0]], [[3, 3], [1, 1]])
print(r1, r2, r3)
assert r1 == (5, 7)
assert r2 == [(1, 0, 3), (1, 0, 9), (1, 2, 5), (2, 1, 0)]
assert r3 == [1]

(5, 7) [(1, 0, 3), (1, 0, 9), (1, 2, 5), (2, 1, 0)] [1]


### Đáp án Bài 2

Bảng khoảng cách: công nhân 0 `(0,0)`: `[1, 4, 3, 10]`; công nhân 1 `(3,0)`: `[2, 1, 6, 7]`; công nhân 2 `(1,3)`: `[3, 4, 1, 6]`. Khoảng cách nhỏ nhất là `1` với các cặp `(0,0)`, `(1,1)`, `(2,2)`; chúng không tranh nhau nên đều được gán. Kết quả `[0, 1, 2]`.

In [34]:
hand_answer_solution = [0, 1, 2]
w, b = [[0, 0], [3, 0], [1, 3]], [[1, 0], [3, 1], [0, 3], [5, 5]]
assert hand_answer_solution == sol(w, b) == assign_brute(w, b)
assert [[manhattan(p, q) for q in b] for p in w] == [[1, 4, 3, 10], [2, 1, 6, 7], [3, 4, 1, 6]]
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [35]:
def exercise_3_solution(workers, bikes):
    n, m = len(workers), len(bikes)
    buckets = [[] for _ in range(1999)]
    for i, (wx, wy) in enumerate(workers):
        for j, (bx, by) in enumerate(bikes):
            buckets[abs(wx - bx) + abs(wy - by)].append(i * m + j)
    ans = [-1] * n
    used = [False] * m
    for bucket in buckets:
        for code_ in bucket:
            i, j = divmod(code_, m)
            if ans[i] == -1 and not used[j]:
                ans[i] = j
                used[j] = True
    return ans

for w, b, e in examples:
    assert exercise_3_solution(w, b) == e
rng = random.Random(3)
for _ in range(300):
    w, b = random_instance(rng, 4, 6, 5)
    assert exercise_3_solution(w, b) == assign_brute(w, b)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [36]:
def exercise_4_solution(workers, bikes):
    n, m = len(workers), len(bikes)
    pairs = sorted((manhattan(workers[i], bikes[j]), i, j) for i in range(n) for j in range(m))
    ans = [-1] * n
    used = [False] * m
    done = 0
    for _, i, j in pairs:
        if ans[i] == -1 and not used[j]:
            ans[i], used[j] = j, True
            done += 1
            if done == n:
                break
    return ans

for w, b, e in examples:
    assert exercise_4_solution(w, b) == e
rng = random.Random(4)
for _ in range(300):
    w, b = random_instance(rng, 4, 6, 5)
    assert exercise_4_solution(w, b) == assign_brute(w, b)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [37]:
def exercise_5_solution(workers, bikes):
    n, m = len(workers), len(bikes)
    buckets = [[] for _ in range(1999)]
    for i, (wx, wy) in enumerate(workers):
        for j, (bx, by) in enumerate(bikes):
            buckets[abs(wx - bx) + abs(wy - by)].append((i, j))
    ans, used, order = [-1] * n, [False] * m, []
    for bucket in buckets:
        for i, j in bucket:
            if ans[i] == -1 and not used[j]:
                ans[i], used[j] = j, True
                order.append((i, j))
    return order

def order_oracle_solution(workers, bikes):
    n, m = len(workers), len(bikes)
    ans, used, order = [-1] * n, [False] * m, []
    for _ in range(n):
        _, i, j = min((manhattan(workers[i], bikes[j]), i, j) for i in range(n) if ans[i] == -1 for j in range(m) if not used[j])
        ans[i], used[j] = j, True
        order.append((i, j))
    return order

assert exercise_5_solution(*examples[1][:2]) == [(0, 0), (1, 2), (2, 1)]
rng = random.Random(5)
for _ in range(300):
    w, b = random_instance(rng, 4, 6, 5)
    assert exercise_5_solution(w, b) == order_oracle_solution(w, b)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (xếp xô theo khoảng cách)

```python
class Solution:
    def assignBikes(self, workers: List[List[int]], bikes: List[List[int]]) -> List[int]:
        n, m = len(workers), len(bikes)
        buckets = [[] for _ in range(1999)]               # khoảng cách Manhattan nằm trong 0..1998
        for i, (wx, wy) in enumerate(workers):
            for j, (bx, by) in enumerate(bikes):
                buckets[abs(wx - bx) + abs(wy - by)].append(i * m + j)
        ans = [-1] * n
        used = [False] * m
        done = 0
        for bucket in buckets:
            for code in bucket:
                i, j = divmod(code, m)
                if ans[i] == -1 and not used[j]:
                    ans[i] = j
                    used[j] = True
                    done += 1
                    if done == n:
                        return ans
        return ans
```

Công thức nhớ nhanh: **mọi cặp xếp theo `(dist, i, j)`; cặp nào còn cả công nhân lẫn xe rảnh thì gán**.

### Checklist trước khi submit

- [ ] Khóa là `(dist, công nhân, xe)` theo đúng thứ tự phá hòa của đề?
- [ ] Cặp được sinh theo thứ tự `i` ngoài, `j` trong (nếu dựa vào sắp xếp ổn định / xô)?
- [ ] Dùng khoảng cách **Manhattan**, và mảng xô có `1999` phần tử?
- [ ] Kiểm tra cả `ans[i] == -1` và `not used[j]` trước khi gán?
- [ ] Đã thử: `n = 1`, nhiều hòa khoảng cách, khoảng cách `1998`, `n = m = 1000`?